# 11. タイムトラベルと VACUUM - 過去はいつまで残るのか

`10` で `OPTIMIZE` を実行したとき、こう書きました。

> 古いファイルが消えたわけではありません。
> Deltaは「今はこの新しいファイルが有効」と記録し直しただけです。

ここを掘ります。

Deltaは **ファイルを書き換えません。**
更新も削除も、新しいファイルを足して「今はこれが有効」と記録するだけです。
古いファイルはそのまま置かれています。

だから **過去のバージョンをそのまま読めます。** これがタイムトラベルです。
そして放っておくと溜まり続けるので、いつかは捨てることになります。それが `VACUUM` です。

この2つは表裏です。**捨てれば戻れなくなります。**
どこまで戻れるようにしておくか、を決める話でもあります。

このノートブックで確かめること:

1. 変更のたびにバージョンが増えること
2. 過去のバージョンを読む、そこへ戻す
3. 保持期間を縮めると何が起きるか
4. 保持期間をどう決めるか

**前提**: `00_setup` を実行済みであること。`10` を読んでいること。


## 準備


In [1]:
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.profile("free").serverless(True).getOrCreate()

In [2]:
CATALOG = "tech_survey"
TABLE = f"{CATALOG}.silver.timetravel_orders"

## 1. 変更するたびにバージョンが増える

3件入れて、1件の金額を直して、1件消します。ごく普通の操作です。


In [3]:
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")

spark.sql(f"""
    CREATE TABLE {TABLE} (
        order_id INT,
        product STRING,
        amount INT
    )
""")

spark.sql(f"""
    INSERT INTO {TABLE} VALUES
        (1, 'laptop',   150000),
        (2, 'monitor',   40000),
        (3, 'keyboard',  12000)
""")

# 1件の金額を直す
spark.sql(f"UPDATE {TABLE} SET amount = 160000 WHERE order_id = 1")

# 1件消す
spark.sql(f"DELETE FROM {TABLE} WHERE order_id = 3")

# 今の中身を確認する
display(spark.table(TABLE).orderBy("order_id"))

,order_id,product,amount
0,1,laptop,160000
1,2,monitor,40000


In [4]:
# 操作のたびにバージョンが1つ増えている
display(spark.sql(f"DESCRIBE HISTORY {TABLE}").select("version", "operation").orderBy("version"))

,version,operation
0,0,CREATE TABLE
1,1,WRITE
2,2,UPDATE
3,3,DELETE
4,4,OPTIMIZE


`CREATE TABLE` / `WRITE` (INSERT) / `UPDATE` / `DELETE` が並んでいます。

ここで大事なのは、**`UPDATE` と `DELETE` でもファイルは消えていない** ことです。

Deltaが `UPDATE` でやっているのは、対象の行を含むファイルを読んで、
直した内容で **新しいファイルを書き、古いほうを「もう使わない」と記録する** ことです。
古いファイルはストレージに残ったままです。`DELETE` も同じです。

### `OPTIMIZE` が勝手に混ざっていたら

履歴に、自分が実行していない `OPTIMIZE` が入っていることがあります。
`10` で触れた **Predictive Optimization** が、裏で走っているためです。

つまり **バージョン番号は連番で予測できません。**
「INSERTしたから次は1番」と決め打ちすると、自動処理が1つ挟まっただけでずれます。

なので、番号は **履歴から引く** ことにします。


In [5]:
# バージョン番号は決め打ちせず、操作の種類から引く
history = spark.sql(f"DESCRIBE HISTORY {TABLE}")

V_INSERT = history.filter("operation = 'WRITE'").select("version").first()[0]
V_UPDATE = history.filter("operation = 'UPDATE'").select("version").first()[0]

print("INSERT した時点:", V_INSERT)
print("UPDATE した時点:", V_UPDATE)

INSERT した時点: 1
UPDATE した時点: 2


## 2. 過去を読む

ファイルが残っているので、過去のバージョンをそのまま読めます。
`VERSION AS OF` にバージョン番号を渡します。


In [6]:
# INSERT した直後。まだ直しても消してもいない状態
display(spark.sql(f"SELECT * FROM {TABLE} VERSION AS OF {V_INSERT} ORDER BY order_id"))

,order_id,product,amount
0,1,laptop,150000
1,2,monitor,40000
2,3,keyboard,12000


In [7]:
from datetime import UTC

# 番号ではなく時刻でも指定できる。ただしタイムゾーンの変換が要る (理由は下)
ts_local = history.filter(f"version = {V_INSERT}").select("timestamp").first()[0]
ts_utc = ts_local.astimezone(UTC)

print("Python側で受け取った値 (ローカル時刻):", ts_local)
print("サーバーに渡す値 (UTC)            :", ts_utc)

display(spark.sql(f"SELECT * FROM {TABLE} TIMESTAMP AS OF '{ts_utc:%Y-%m-%d %H:%M:%S.%f}' ORDER BY order_id"))

Python側で受け取った値 (ローカル時刻): 2026-09-16 10:16:22
サーバーに渡す値 (UTC)            : 2026-09-16 01:16:22+00:00


,order_id,product,amount
0,1,laptop,150000
1,2,monitor,40000
2,3,keyboard,12000


### タイムスタンプを渡すときの落とし穴

`astimezone` を挟んでいるのには理由があります。

`DESCRIBE HISTORY` の `timestamp` は、Python側では **タイムゾーン情報を持たない `datetime`** として返ってきます。
しかも値は **こちらのローカル時刻 (JST)** に変換済みです。

一方、Databricks側のセッションのタイムゾーンは **UTC** です
(`spark.conf.get("spark.sql.session.timeZone")` で確認できます)。

なので、受け取った値をそのまま文字列にしてSQLへ埋めると、サーバーは **JSTの壁時計時刻をUTCとして読みます。**
9時間先を指すことになり、こう怒られます。

```
[DELTA_TIMESTAMP_GREATER_THAN_COMMIT] Timestamp 2026-09-16 09:48:26.0
is after the table's latest version (2026-09-16 00:48:43.0)
```

エラーになるのはまだ親切なほうです。
ずれた先に別のバージョンが存在する場合は、**エラーにならず、意図と違う時点の結果が静かに返ります。**

番号で指定するときは、この問題は起きません。
**時刻で指定するときだけ、どちらのタイムゾーンで解釈されるかを意識する必要があります。**

実務で時刻指定を使いたい場面は多いはずです。
「昨日の朝の時点ではどうだったか」を調べるとき、バージョン番号は分からないからです。


## 3. 戻す

読むだけでなく、テーブルそのものを過去の状態に戻せます。`RESTORE` です。


In [8]:
# 自動OPTIMIZEと書き込みが重なると ConcurrentWriteException になることがある
# その場合はこのセルをもう一度実行する
spark.sql(f"RESTORE TABLE {TABLE} TO VERSION AS OF {V_INSERT}")

display(spark.table(TABLE).orderBy("order_id"))

,order_id,product,amount
0,1,laptop,150000
1,2,monitor,40000
2,3,keyboard,12000


In [9]:
# 戻したこと自体も、新しいバージョンとして記録される
display(spark.sql(f"DESCRIBE HISTORY {TABLE}").select("version", "operation").orderBy("version"))

,version,operation
0,0,CREATE TABLE
1,1,WRITE
2,2,UPDATE
3,3,DELETE
4,4,OPTIMIZE
5,5,RESTORE


中身は INSERT 直後の状態に戻りましたが、**履歴は消えていません。**
`RESTORE` が新しいバージョンとして足されています。

つまり「戻した」ことも取り消せます。履歴は積み上がる一方で、巻き戻りません。
操作を打ち消すのではなく、**打ち消す操作を足している** と考えると分かりやすいです。

### `ConcurrentWriteException` が出たら

もし `ConcurrentWriteException` で失敗したら、もう一度実行すれば通ります。

Deltaは **楽観的並行制御** で動いています。
「たぶん誰も触っていないだろう」と思って処理を進め、
書き込む直前に「本当に誰も触っていなかったか」を確認します。
その間に他の誰かが書いていたら、後から来たほうが失敗します。

ここでの「他の誰か」は、たいてい自動 `OPTIMIZE` です。
自分ひとりで触っているつもりでも、裏では別の処理が動いている、という例になります。


## 4. 保持期間を縮めてみる

ここまでが「残っているから戻れる」話でした。裏返すと、**ストレージを使い続けている** ということです。

`VACUUM` は、**今のテーブルが使っていないファイル** を物理的に削除します。
どこまで残すかは、テーブルの設定 `delta.deletedFileRetentionDuration` で決まります。既定は **7日** です。

7日待つわけにはいかないので、この設定を **0時間** にしてから `VACUUM` します。

> **注意**: ここから先は戻せません。実験用のテーブルだから短くしています。
> 本番のテーブルで保持期間を0にしてはいけません。

なお、`VACUUM テーブル RETAIN 0 HOURS` と書く方法もありますが、
**サーバーレスでは使えません。** これは既定で拒否され、
解除するための設定 (`spark.databricks.delta.retentionDurationCheck.enabled`) が
サーバーレスでは変更できないためです。テーブル側の設定を変えるほうが確実です。


In [10]:
# テーブルの保持期間を0にする
spark.sql(f"""
    ALTER TABLE {TABLE}
    SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours')
""")

# DRY RUN = 実際には消さず、消える対象だけを出す。まずこれで確認する
display(spark.sql(f"VACUUM {TABLE} DRY RUN"))

,path
0,
1,
2,
3,
4,


In [11]:
# ここから戻せない
spark.sql(f"VACUUM {TABLE}")

# 今のテーブルは無事
display(spark.table(TABLE).orderBy("order_id"))

,order_id,product,amount
0,1,laptop,150000
1,2,monitor,40000
2,3,keyboard,12000


In [12]:
# 過去のバージョンを指定してみる
# エラーメッセージを読みたいので、例外を捕まえて表示する
try:
    display(spark.sql(f"SELECT * FROM {TABLE} VERSION AS OF {V_UPDATE} ORDER BY order_id"))
except Exception as e:
    print(type(e).__name__)
    print(str(e)[:300])

AnalysisException
[DELTA_UNSUPPORTED_TIME_TRAVEL_BEYOND_DELETED_FILE_RETENTION_DURATION] Cannot time travel beyond delta.deletedFileRetentionDuration (0 HOURS) set on the table.

JVM stacktrace:
com.databricks.sql.transaction.tahoe.DeltaAnalysisException
	at com.databricks.sql.transaction.tahoe.DeltaErrorsBase.timeTr


`DELTA_UNSUPPORTED_TIME_TRAVEL_BEYOND_DELETED_FILE_RETENTION_DURATION` が出たはずです。

注目してほしいのは、エラーの理由が **「ファイルが無い」ではない** ことです。
**「保持期間を超えている」** と言っています。

Deltaはファイルを探しに行く前に、設定を見て断っています。
保持期間を0にした時点で、**戻れる範囲が0になった** わけです。

一方、**今のテーブルは普通に読めています。** 現在有効なファイルは消されないからです。
消えたのは「もう使っていないファイル」だけで、`DRY RUN` に出ていたものがそれです。

履歴の一覧からは、過去のバージョンも消えていません。
**記録は残るのに、そこへは行けない** という状態になります。
`DESCRIBE HISTORY` に見えるからといって戻れるとは限らない、ということです。


## 5. 保持期間をどう決めるか

既定は7日です。長くするか短くするかは、次のトレードオフになります。

| | 長くする | 短くする |
|---|---|---|
| 戻せる範囲 | 広い | 狭い |
| ストレージ費用 | 増える | 減る |
| 誤削除からの復旧 | 間に合う | 間に合わないことがある |

決め方は **「気づくまでにかかる時間」から逆算する** のが素直です。

データがおかしくなったとき、気づくのが日次のチェックなら最悪でも1〜2日。
月次の締めで初めて気づくなら、1か月以上必要になります。
**気づく前に消えてしまうなら、タイムトラベルは無いのと同じ** です。

7日のままで困らないことが多いですが、次の場合は延ばす価値があります。

- 月次でしか検算しない集計テーブル
- 誤ったバッチを流したときに、戻せることが最後の砦になるテーブル

逆に短くしてよいのは **いつでも作り直せるテーブル** です。
Bronzeから何度でも再生成できるSilver/Goldなら、過去を長く持つ意味は薄くなります。

### ストリーミングで読んでいるテーブルには注意

`01` で見たように、ストリーミングはチェックポイントに「どのファイルまで読んだか」を記録しています。
その状態で `VACUUM` がファイルを消すと、**まだ読んでいないファイルが消える** ことがあります。

ストリームが止まっていた間に `VACUUM` が走った場合などが該当します。
再開しようとしても元ファイルが無いので、そのぶんは取りこぼします。

保持期間は、**一番長く止まりうるストリームより長く** 取る必要があります。


## 考えてみる

- 履歴には残っているのに読めないバージョンがありました。履歴のほうも消せるのでしょうか
- `10` の `OPTIMIZE` を実行した後に `VACUUM` すると、何が消えますか
- テーブルを `DROP` した場合、タイムトラベルで戻せるでしょうか


### 答え

**Q1. 履歴は消せるのか**

履歴 (トランザクションログ) にも保持期間があり、既定は **30日** です (`delta.logRetentionDuration`)。
これを過ぎたものはログからも消えます。

ファイルの保持 (既定7日) とログの保持 (既定30日) は **別々の設定** です。
だから今回のように「ログには残っているが、そこへは戻れない」というズレが起きます。

戻れる範囲を決めているのは、短いほうである **ファイルの保持期間** です。
ログだけ長くしても戻せる範囲は広がりません。

**Q2. `OPTIMIZE` の後に `VACUUM` すると**

**まとめる前の小さいファイルが消えます。**

`OPTIMIZE` は中身を変えずに大きいファイルへ書き直す操作なので、
直後は「古い小さいファイル」と「新しい大きいファイル」の両方がストレージにあります。
`VACUUM` して初めて、古いほうが消えて容量が減ります。

つまり **`OPTIMIZE` だけではストレージは減りません。**
`10` で「サイズがほとんど減らない」と書いたのは、これが理由です。

**Q3. `DROP` したら**

**タイムトラベルでは戻せません。** テーブルが存在していることが前提の機能だからです。

Unity Catalogの管理テーブルには `UNDROP TABLE` があり、
一定期間内なら消したテーブル自体を復活させられます。
ただしこれはタイムトラベルとは別の仕組みです。


## 後片付け

このノートブックで作ったものを消したいときだけ、コメントを外して実行します。


In [13]:
# spark.sql(f"DROP TABLE IF EXISTS {TABLE}")